In [3]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

#Load & Clean
df = pd.read_excel('healthcare_realistic_dataset_515_rows.xlsx')
df = df.dropna(subset=['Treatment_Cost'])
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df = df[df['Age'].between(0, 120)]

bins = [17, 30, 45, 60, 120]
labels = ['18–30', '31–45', '46–60', '60+']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels)

# Q1: Disease with highest total treatment cost
print("=" * 60)
print("Q1: Total Treatment Cost by Disease")
print("=" * 60)
q1 = df.groupby('Disease')['Treatment_Cost'].sum().sort_values(ascending=False)
print(q1.apply(lambda x: f"₹{x/1e6:.2f}M"))
print(f"\n✅ Answer: {q1.index[0]} has the highest total cost: ₹{q1.iloc[0]/1e6:.2f}M\n")

# Q2: Department with highest revenue
print("=" * 60)
print("Q2: Revenue by Department")
print("=" * 60)
q2 = df.groupby('Department')['Treatment_Cost'].sum().sort_values(ascending=False)
print(q2.apply(lambda x: f"₹{x/1e6:.2f}M"))
print(f"\n✅ Answer: {q2.index[0]} generates the highest revenue: ₹{q2.iloc[0]/1e6:.2f}M\n")

# Q3: Average treatment cost by disease and insurance status
print("=" * 60)
print("Q3: Avg Treatment Cost by Disease & Insurance Status")
print("=" * 60)
q3 = df.groupby(['Disease', 'Insurance_Status'])['Treatment_Cost'].mean().round(2)
print(q3.apply(lambda x: f"₹{x:,.0f}"))
print()

# Q4: % costs covered by insured patients
print("=" * 60)
print("Q4: % of Total Treatment Costs by Insurance Status")
print("=" * 60)
total = df['Treatment_Cost'].sum()
insured = df[df['Insurance_Status'] == 'Yes']['Treatment_Cost'].sum()
uninsured = df[df['Insurance_Status'] == 'No']['Treatment_Cost'].sum()
print(f"Total Cost     : ₹{total/1e6:.2f}M")
print(f"Insured Cost   : ₹{insured/1e6:.2f}M ({insured/total*100:.2f}%)")
print(f"Uninsured Cost : ₹{uninsured/1e6:.2f}M ({uninsured/total*100:.2f}%)")
print(f"\n✅ Answer: Insured patients cover {insured/total*100:.2f}% of total treatment costs\n")

# Q5: City with highest average treatment cost
print("=" * 60)
print("Q5: Average Treatment Cost per Patient by City")
print("=" * 60)
q5 = df.groupby('City')['Treatment_Cost'].mean().sort_values(ascending=False)
print(q5.apply(lambda x: f"₹{x/1e3:.1f}K"))
print(f"\n✅ Answer: {q5.index[0]} has the highest avg cost: ₹{q5.iloc[0]/1e3:.1f}K/patient\n")

# Q6: Age group with highest healthcare expenditure
print("=" * 60)
print("Q6: Healthcare Expenditure by Age Group")
print("=" * 60)
q6 = df.groupby('Age_Group', observed=True)['Treatment_Cost'].sum().sort_values(ascending=False)
print(q6.apply(lambda x: f"₹{x/1e6:.2f}M"))
print(f"\n✅ Answer: {q6.index[0]} age group has highest expenditure: ₹{q6.iloc[0]/1e6:.2f}M\n")

# Q7: Most common disease per age group
print("=" * 60)
print("Q7: Most Common Disease per Age Group")
print("=" * 60)
q7 = df.groupby(['Age_Group', 'Disease'], observed=True).size().reset_index(name='Count')
top_disease = q7.loc[q7.groupby('Age_Group', observed=True)['Count'].idxmax()]
print(top_disease.to_string(index=False))
print("\nFull disease-age matrix:")
matrix = q7.pivot_table(index='Disease', columns='Age_Group', values='Count', aggfunc='sum', fill_value=0)
print(matrix)
print()

# Q8: Treatment cost by gender
print("=" * 60)
print("Q8: Treatment Cost by Gender")
print("=" * 60)
q8 = df.groupby('Gender')['Treatment_Cost'].agg(['mean','sum','count']).round(2)
q8.columns = ['Avg Cost', 'Total Cost', 'Patients']
print(q8.assign(**{'Avg Cost': q8['Avg Cost'].apply(lambda x: f"₹{x/1e3:.1f}K"),
                   'Total Cost': q8['Total Cost'].apply(lambda x: f"₹{x/1e6:.2f}M")}))
print()

# Q9: Blood group with highest critical outcomes
print("=" * 60)
print("Q9: Blood Group - Critical Outcome Rate")
print("=" * 60)
q9_all = df.groupby('Blood_Group').size().rename('Total')
q9_crit = df[df['Outcome'] == 'Critical'].groupby('Blood_Group').size().rename('Critical')
q9 = pd.concat([q9_all, q9_crit], axis=1).fillna(0)
q9['Critical_Rate_%'] = (q9['Critical'] / q9['Total'] * 100).round(2)
q9 = q9.sort_values('Critical_Rate_%', ascending=False)
print(q9)
print(f"\n✅ Answer: {q9.index[0]} has the highest critical rate: {q9.iloc[0]['Critical_Rate_%']}%\n")

print("=" * 60)
print("Analysis Complete!")
print("=" * 60)

Q1: Total Treatment Cost by Disease
Disease
Asthma           ₹8.25M
Arthritis        ₹7.74M
Heart Disease    ₹7.32M
Pneumonia        ₹6.69M
Diabetes         ₹6.55M
Hypertension     ₹5.93M
Migraine         ₹5.83M
COVID-19         ₹5.73M
Name: Treatment_Cost, dtype: object

✅ Answer: Asthma has the highest total cost: ₹8.25M

Q2: Revenue by Department
Department
Neurology           ₹11.52M
Oncology            ₹11.09M
General Medicine     ₹9.01M
Pulmonology          ₹8.53M
Cardiology           ₹8.34M
Orthopedics          ₹8.19M
Name: Treatment_Cost, dtype: object

✅ Answer: Neurology generates the highest revenue: ₹11.52M

Q3: Avg Treatment Cost by Disease & Insurance Status
Disease        Insurance_Status
Arthritis      No                  ₹132,297
               Yes                 ₹134,243
Asthma         No                  ₹116,354
               Yes                 ₹119,874
COVID-19       No                  ₹116,347
               Yes                 ₹132,643
Diabetes       No      